# GeoLens + leafmap

[GeoLens](https://github.com/geolens-io/geolens) is a self-hosted spatial data
hub: a catalog, search, and open standards over data that stays on your own
infrastructure. It serves OGC API Features, OGC API Records (catalog search),
STAC 1.0, MVT vector tiles, and raster tiles baked by
[TiTiler](https://developmentseed.org/titiler/). Every one of them is plain
HTTP, so nothing here needs a GeoLens-specific client.

This notebook reads a live instance into [leafmap](https://leafmap.org/):
search the catalog, load a feature collection, push a filter to the server
with CQL2, and drop raster tiles on the map. `GEOLENS` below defaults to the
public demo, which needs no account and no key. Pointing this at your own
instance takes more than changing that one line: the dataset ids further
down (`STATIONS`, `LINES`, `DEM`) belong to the demo's catalog, so list
`/api/collections` on your own instance and substitute yours; a private
instance also needs `GEOLENS_API_KEY` set, and its own raster tiles need a
signed tile token rather than the bare template (see section 4). The
assertions, the map centers and the raster probe tile are calibrated to
the demo's own data too; they're what makes this notebook self-checking
rather than something your own catalog needs to match, so expect to loosen
or drop them once you're pointed elsewhere.


In [ ]:
# Pinned so a leafmap or geopandas upgrade doesn't change what this notebook
# does out from under you. Safe to re-run: pip skips anything already at the
# pinned version.
%pip install -q leafmap==0.63.1 geopandas==1.1.4 requests==2.33.1


In [ ]:
import io
import os
import time

import geopandas as gpd
import leafmap.foliumap as leafmap
import requests

# Point this at your own instance to use your own catalog. The public demo
# answers every route below anonymously.
GEOLENS = "https://demo.getgeolens.com"
API = f"{GEOLENS}/api"

# Public datasets need no credentials, so the demo works with none set. For
# a private instance, export GEOLENS_API_KEY and every request below sends
# it as an X-Api-Key header.
API_KEY = os.environ.get("GEOLENS_API_KEY")
HEADERS = {"X-Api-Key": API_KEY} if API_KEY else {}

# The demo is one shared machine on the public internet, so a request
# occasionally times out or comes back 502 with nothing wrong at either end.
# python/analyze.py retries those same failures against this same demo;
# everything below goes through get_with_retry() instead of requests.get()
# for the same reason. Anything else in the 4xx range is a bad request, and
# retrying it just asks the same wrong question again.
RETRY_STATUS = {429, 500, 502, 503, 504}


def get_with_retry(url: str, attempts: int = 3, backoff: float = 1.0, **kwargs) -> requests.Response:
    kwargs.setdefault("timeout", 30)
    kwargs.setdefault("headers", HEADERS)
    for attempt in range(attempts):
        try:
            resp = requests.get(url, **kwargs)
        except (requests.exceptions.ConnectionError, requests.exceptions.Timeout) as exc:
            reason = type(exc).__name__
        else:
            if resp.status_code not in RETRY_STATUS:
                resp.raise_for_status()
                return resp
            reason = f"HTTP {resp.status_code}"
        if attempt + 1 < attempts:
            print(f"  {reason}, retrying")
            time.sleep(backoff * 2**attempt)
    raise RuntimeError(f"{url} failed {attempts} times, last {reason}")


## 1. Discover datasets: search the catalog

`GET /api/search/datasets/` is OGC API Records underneath: it runs the phrase
against embeddings of each record's title, description and keywords, so a
hit doesn't need to share a word with the query. On the demo, this exact
phrase matches the meteorite landings dataset with none of its five words
in that dataset's title.


In [ ]:
resp = get_with_retry(f"{API}/search/datasets/", params={"q": "space rocks that fell to earth", "limit": 5})
results = resp.json()

print(f"{results['numberMatched']} match(es), meaning-matched rather than keyword-matched:")
for feature in results["features"]:
    props = feature["properties"]
    print(f"  {feature['id']}  {props['title']!r}  ({props['record_type']})")

# The meteorite dataset should be the top hit. If search regresses to an
# empty or unrelated result, catch it here rather than let a printed-but-
# wrong result look like a pass.
top_hit = results["features"][0]["properties"]["title"] if results["features"] else None
assert top_hit == "Meteorite Landings (Meteoritical Society)", f"expected the meteorite dataset first, got {top_hit!r}"


## 2. Load a vector collection over OGC API Features

`/api/collections/{id}/items` returns plain GeoJSON, so geopandas can read it
directly. The two collections below are the demo's NYC subway layers, small
enough (496 stations, 29 service lines) that one `limit=2000` request holds
the whole thing; a layer with more rows would need to follow the `next` link
GeoLens paginates with, the same way
[`python/analyze.py`](../python/analyze.py) does.

One gotcha worth naming: `gpd.read_file(url)` looks tempting, but geopandas
issues its own pre-flight request with the bare `urllib` user agent to sniff
the format, and the demo's CDN answers that agent with 403 while answering
`requests` (and a browser, and GDAL/curl) with 200. Fetching the bytes
ourselves sidesteps the sniff.


In [ ]:
STATIONS = "724bf894-dc1a-418c-abc6-555798c44d7c"  # NYC Subway Stations (MTA)
LINES = "de602fbe-8b30-4755-924f-c9e7fd9613b6"      # NYC Subway Lines (MTA)


def read_collection(collection_id: str, **params) -> gpd.GeoDataFrame:
    '''Read one page of an OGC API - Features collection into a GeoDataFrame.'''
    resp = get_with_retry(
        f"{API}/collections/{collection_id}/items",
        params={"limit": 2000, **params},
    )
    return gpd.read_file(io.BytesIO(resp.content))


stations = read_collection(STATIONS)
lines = read_collection(LINES)

assert len(stations) > 400 and len(lines) > 20, "expected the full demo subway layers"
print(f"{len(stations)} stations, {len(lines)} service lines, both in {stations.crs}")


In [ ]:
m = leafmap.Map(center=[40.75, -73.98], zoom=11)
m.add_gdf(lines, layer_name="Subway lines", style={"color": "#4da3ff", "weight": 2})
m.add_gdf(stations, layer_name="Subway stations", style={"color": "#ffd166", "radius": 3, "fillOpacity": 0.9})
m


## 3. Filter server-side with CQL2

[OGC API Features Part 3](https://docs.ogc.org/is/19-079r2/19-079r2.html)
lets a client hand the server a filter instead of downloading everything and
filtering locally. GeoLens evaluates `filter=` server-side against the
`datasets` collection: the catalog itself, one record per dataset. That's
what the query below narrows, and it's a different question from filtering
the *rows inside* a dataset (the stations and lines above). This instance
answers CQL2 on the catalog today, and not yet on a dataset's own feature
collection, so the check below confirms the conformance class before relying
on it rather than assuming a specific version.


In [ ]:
conformance = get_with_retry(f"{API}/conformance").json()["conformsTo"]
if "http://www.opengis.net/spec/cql2/1.0/conf/basic-cql2" not in conformance:
    print("This instance doesn't advertise CQL2 support; skipping.")
else:
    cql2_filter = "title LIKE '%Subway%'"
    resp = get_with_retry(f"{API}/collections/datasets/items", params={"filter": cql2_filter})
    matches = gpd.read_file(io.BytesIO(resp.content))

    print(f"CQL2 filter {cql2_filter!r} matched {len(matches)} of the catalog's ~30 datasets:")
    print(matches[["title"]].to_string(index=False))
    assert set(matches["title"]) == {"NYC Subway Lines (MTA)", "NYC Subway Stations (MTA)"}


## 4. Raster: COG tiles through TiTiler

GeoLens ingests a raster once and bakes it into XYZ tiles with TiTiler, so a
client never has to open the COG itself: point a tile layer at the template
and every viewer past that gets pixels, not a multi-hundred-megabyte file.
The demo publishes a DEM over the Matterhorn as one example. Outside the
DEM's footprint the route answers `204` by design (that's the server saying
there's no tile there, not a broken one), so the map below opens centered on
the peak.

The public DEM below needs no credential, but a private raster on your own
instance does, and the browser fetching these tiles has no way to attach
an `X-Api-Key` header to a plain URL template. `GET /api/tiles/token/<id>/`
mints a short-lived signed URL for exactly this case; for a raster dataset
it returns a `tile_url` with the signature already in the query string,
root-relative the way a browser's own `fetch` would return one, so a
private DEM means prefixing it with `GEOLENS` and minting a fresh one
before each expires rather than authenticating the tile requests some
other way.


In [ ]:
DEM = "6f03bafa-34b3-4902-9351-40ce09a8181f"  # swissALTI3D Matterhorn DEM (2m mosaic)
tile_url = f"{GEOLENS}/raster-tiles/{DEM}/tiles/{{z}}/{{x}}/{{y}}.png"

# Confirm the route is live before wiring it into the map.
probe = get_with_retry(f"{GEOLENS}/raster-tiles/{DEM}/tiles/12/2135/1457.png")
assert probe.headers["content-type"] == "image/png"

m2 = leafmap.Map(center=[45.976, 7.658], zoom=12)
m2.add_tile_layer(url=tile_url, name="Matterhorn DEM", attribution="GeoLens demo")
m2


## 5. Optional: segment the DEM with samgeo

[`segment-geospatial`](https://samgeo.gishub.org/) runs Meta's Segment
Anything Model over a georeferenced raster. It needs `torch` and a
multi-hundred-megabyte model checkpoint, neither of which belongs in a
notebook that is supposed to run anywhere in a few seconds, so this cell is
off by default. Flip the flag and install the extra once, and it turns the
tile layer above into a local GeoTIFF and segments it.


In [ ]:
RUN_SEGMENTATION = False  # pip install segment-geospatial torch, then flip this on

if RUN_SEGMENTATION:
    from samgeo import SamGeo

    # samgeo segments a raster file, not a live tile source, so pull the tiles
    # over the DEM's footprint into a local GeoTIFF first.
    dem_bbox = [7.60, 45.95, 7.72, 46.01]
    leafmap.tms_to_geotiff("matterhorn.tif", dem_bbox, zoom=14, source=tile_url, to_cog=True)

    sam = SamGeo(model_type="vit_h", automatic=True)
    sam.generate("matterhorn.tif", output="matterhorn_segments.tif")
    m2.add_raster("matterhorn_segments.tif", layer_name="Segments", opacity=0.6)
    m2
else:
    print("Skipping: RUN_SEGMENTATION is False. See the README for what this needs.")


---

Read further: the [API reference](https://docs.getgeolens.com/guides/api/ogc/)
covers every route above, [`qgis/`](../qgis/) reads the same catalog from
desktop GIS, and [`python/analyze.py`](../python/analyze.py) does a
comparable spatial join in plain GeoPandas. If GeoLens is useful to you,
[star it on GitHub](https://github.com/geolens-io/geolens). That's how most
people find it.
